# Python for AI — Class 8
### Recursion and decorators

Class 7 treated functions as values you can pass around — that's what `lambda` gave you. Today functions do more: they call *themselves* (recursion) and *wrap* other functions (decorators).

Each one comes with real business examples — a P&L, refund permissions, an audit log, FastAPI routes. Generators pick up in Class 9.

**How to use this notebook:** run each cell with the play button, or `Shift + Enter`.

Cells marked **BREAKS ON PURPOSE** are *supposed* to show a red error. Cells marked **WRONG ON PURPOSE** run fine and give the *wrong answer* — those are the dangerous ones. Don't fix either before class; that is the lesson.

# 1. Recursion — a function that calls itself

A recursive function does its job by handing a **smaller version of the same job** to itself. ("Job" here just means whatever you've asked the function to do — count down from 3, add up 1 to 10, reverse a word.)

Every recursive function needs two parts:

1. A **base case** — the smallest version of the job, simple enough to answer directly, with no further recursive call.
2. A **recursive case** — the function calling itself with a smaller piece of the job, moving toward the base case.

Miss the base case, and it's Day 3's infinite loop again — except this time it's an infinite chain of function calls.

### Start simple: a countdown

In [5]:
def countdown(n):
    if n == 0:                 # base case - stop here
        print("Done!")
        return
    print(n)
    countdown(n - 1)           # recursive case - the same job, one smaller

countdown(3)

3
2
1
Done!


Follow it call by call:

```
countdown(3)  prints 3, then calls countdown(2)
countdown(2)  prints 2, then calls countdown(1)
countdown(1)  prints 1, then calls countdown(0)
countdown(0)  base case: prints "Done!" and stops
```

Each call does one small piece of the work — print one number — then hands the rest of the job to a smaller copy of itself.

### Adding up 1 to n

In [6]:
def sum_to(n):
    if n == 0:                  # base case - the sum of nothing is 0
        return 0
    return n + sum_to(n - 1)    # n, plus the sum of everything below it

print(sum_to(3))
print(sum_to(10))

6
55


```
sum_to(3) = 3 + sum_to(2)
          = 3 + 2 + sum_to(1)
          = 3 + 2 + 1 + sum_to(0)
          = 3 + 2 + 1 + 0
          = 6
```

### Reversing a word

In [7]:
def reverse(text):
    if text == "":                          # base case - nothing left to reverse
        return ""
    return reverse(text[1:]) + text[0]      # reverse the rest, then put the first letter last



print(reverse("hello"))

olleh


Trace `reverse("cat")`. Each call sets its **first** letter aside and asks a smaller copy of itself to reverse the rest:

```
reverse("cat") = reverse("at") + "c"
               = (reverse("t") + "a") + "c"
               = ((reverse("") + "t") + "a") + "c"
               = (("" + "t") + "a") + "c"          <- base case: reverse("") returns ""
               = ("t" + "a") + "c"                 <- reverse("t") returns "t"
               = "ta" + "c"                        <- reverse("at") returns "ta"
               = "tac"                             <- reverse("cat") returns "tac"
```

Nothing gets joined on the way down — every call is paused, waiting. Only once the base case hands back `""` do the answers travel back up, and each call sticks the letter it set aside onto the **end**. That's why the first letter, `c`, ends up last.

### The classic: factorial

`5!` ("5 factorial") means `5 × 4 × 3 × 2 × 1`. It has exactly the same shape as `sum_to` — multiply instead of add, and the base case returns `1` instead of `0`.

In [8]:
def factorial(n):
    if n == 0:                     # base case - stops the recursion
        return 1
    return n * factorial(n - 1)    # recursive case - a smaller job

print(factorial(5))

120


Trace `factorial(3)` by hand — it has to go all the way down to the base case before anything can be multiplied:

```
factorial(3)
= 3 * factorial(2)
    = 3 * (2 * factorial(1))
        = 3 * (2 * (1 * factorial(0)))
            = 3 * (2 * (1 * 1))         <- base case reached, returns 1
        = 3 * (2 * 1)                   <- factorial(1) returns 1
    = 3 * 2                             <- factorial(2) returns 2
= 6                                     <- factorial(3) returns 6
```

### The call stack

Each call to `factorial` pauses and waits for the call it made to finish, before it can compute its own answer. Python keeps track of every paused, waiting call in a structure called the **call stack** — literally a stack, like Day 4's `append`/`pop`: the most recent call is the first one to finish and get popped off.

Forget the base case, and that stack never stops growing:

In [9]:
# BREAKS ON PURPOSE - no base case, so it never stops calling itself
def broken_factorial(n):
    return n * broken_factorial(n - 1)

broken_factorial(5)

RecursionError: maximum recursion depth exceeded

**RecursionError: maximum recursion depth exceeded.** This is recursion's version of Day 3's infinite loop — except an infinite `while` loop just hangs forever, while Python tracks the call stack's size and refuses to let it grow without limit, so a broken recursion crashes cleanly instead of freezing your program.

### Recursion isn't just for numbers

The "smaller version of the same job" can be a smaller list, a smaller string, a smaller anything.

In [ ]:
def sum_list(numbers):
    if not numbers:                      # base case - an empty list sums to 0
        return 0
    return numbers[0] + sum_list(numbers[1:])   # first item + the sum of the rest

print(sum_list([1, 2, 3, 4, 5]))
print(sum_list([]))

Each call peels off `numbers[0]` and hands the rest, `numbers[1:]`, to itself. The list gets one item shorter every call, so it's guaranteed to eventually hit the base case: an empty list.

In [ ]:
# Explain // operation
# In Python, the `//` operator is used for floor division. 
# It divides two numbers and returns the largest integer less than or equal to the result. 
# This means that it effectively "rounds down" to the nearest whole number.
# Example:
a = 1234 // 10
print(a)  # Output: 123, because 1234 divided by 10 is 123.4, and the floor division returns 123.


123


In [ ]:
def sum_digits(n):
    if n < 10:              # base case - a single digit sums to itself
        return n
    return n % 10 + sum_digits(n // 10)    # last digit + the sum of the rest

print(sum_digits(1234))    # 1 + 2 + 3 + 4

10


### Recursion vs. a loop — the same answer, two shapes

Anything recursion can do, a loop can also do — usually with less overhead, since every recursive call takes up its own space on the call stack, while a loop just keeps reusing the same space.

In [ ]:
def factorial_loop(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

print(factorial_loop(5))

120


Recursion earns its place when a job is *naturally* described in terms of a smaller version of itself — nested data like a business's expense categories is the clearest example, and you'll see one at the end of this section. When a loop and recursion would both work equally well, reach for the loop; it's usually easier to read and cheaper to run.

One more example worth seeing, because it shows recursion's dark side too:

In [ ]:
def fibonacci(n):
    if n <= 1:                              # base case - the first two numbers
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)    # each number is the sum of the two before it

for i in range(8):
    print(fibonacci(i), end=" ")

0 1 1 2 3 5 8 13 

This works, but watch what it's actually doing: `fibonacci(5)` calls `fibonacci(4)` and `fibonacci(3)` — and `fibonacci(4)` *also* calls `fibonacci(3)` separately. The same smaller answers get recalculated over and over, and the number of calls explodes as `n` grows. It's correct, and it's slow. Fixing that (without giving up recursion) is a lesson for later in the course — for now, just recognise that recursion being elegant and recursion being efficient are two different questions.

### Real use case: a P&L with nested expense categories

A **P&L** (profit and loss statement) shows what a business earned, what it spent, and what's left over as profit. Expenses are grouped into categories, which have sub-categories, which can have their own — and every business nests them differently.

You can't know in advance how deep the nesting goes, so you can't write the right number of loops. Recursion doesn't care: every category is handled the same way, however deep. Below, a single expense is just a number, and a category is a dictionary of whatever it contains.

In [ ]:
expenses = {
    "rent": 80000,
    "salaries": {
        "cashiers": 120000,
        "manager": 90000,
    },
    "marketing": {
        "online": {
            "facebook_ads": 25000,
            "google_ads": 15000,
        },
        "flyers": 5000,
    },
    "utilities": 18000,
}

print("Values", expenses.values())

def total_cost(item):
    if type(item) != dict:            # base case - a single expense, just return it
        return item
    amount = 0
    for child in item.values():       # a category - add up everything inside it
        amount += total_cost(child)
    return amount

revenue = 500000
total_expenses = total_cost(expenses)

print("Revenue:       ", revenue)
print("Total expenses:", total_expenses)
print("Profit:        ", revenue - total_expenses)
# print()
print("Marketing alone:", total_cost(expenses["marketing"]))

Values dict_values([80000, {'cashiers': 120000, 'manager': 90000}, {'online': {'facebook_ads': 25000, 'google_ads': 15000}, 'flyers': 5000}, 18000])
Revenue:        500000
Total expenses: 353000
Profit:         147000
Marketing alone: 45000


### Breaking it down, step by step

`total_cost` asks one question about whatever it's handed: **is this a single expense (a number), or a category (a dictionary)?**

- A number is the **base case** — just return it.
- A dictionary is the **recursive case** — call `total_cost` on each thing inside it, and add up what comes back.

Here is every call the program makes. Each indent is a call that its parent is waiting on:

```
total_cost(expenses)                   category -> check its 4 items
│
├── "rent": 80000                      number   -> returns 80000
│
├── "salaries"                         category -> check its 2 items
│   ├── "cashiers": 120000             number   -> returns 120000
│   └── "manager": 90000               number   -> returns 90000
│                                      salaries returns 120000 + 90000 = 210000
│
├── "marketing"                        category -> check its 2 items
│   ├── "online"                       category -> check its 2 items
│   │   ├── "facebook_ads": 25000      number   -> returns 25000
│   │   └── "google_ads": 15000        number   -> returns 15000
│   │                                  online returns 25000 + 15000 = 40000
│   └── "flyers": 5000                 number   -> returns 5000
│                                      marketing returns 40000 + 5000 = 45000
│
└── "utilities": 18000                 number   -> returns 18000

total_cost(expenses) returns 80000 + 210000 + 45000 + 18000 = 353000
```

Which gives the P&L:

| Line | Amount (Rs) |
|---|---|
| Revenue | 500,000 |
| Total expenses | 353,000 |
| **Profit** | **147,000** |

Three things to notice:

1. **The function never needed to know how deep the data goes.** `facebook_ads` sits three levels down, and it was reached without writing a single extra loop. Split Facebook ads into separate campaigns tomorrow — a fourth level — and the same function still works, unchanged.
2. **Every call has its own `amount`.** When `total_cost(salaries)` starts its own `amount = 0`, it doesn't wipe out the `amount` that `total_cost(expenses)` is still building up. That's yesterday's local scope at work: each call gets its own private variables.
3. **The totals are added on the way back up.** Just like `sum_to`, nothing gets added until the deepest calls return. `online` has to finish before `marketing` can, and `marketing` has to finish before the grand total can.

The same function works at **any level** — hand it just `expenses["marketing"]` and you get that category's subtotal, with no extra code.

Plenty of business data has this nested shape: an online store's product categories (Electronics → Phones → Android), a company's org chart, or the JSON an API sends back (Day 18). Whenever data contains smaller copies of itself, recursion is the natural tool.

---
# 2. Decorators — adding behaviour to a function without changing it

A decorator wraps extra behaviour — printing a log, timing, checking a login — around a function, without touching the function's own code. To get there, you need two ideas first.

### Idea 1: a function is a value

A function can be stored in a variable and passed around, like a number or a list. You already did this: `key=lambda ...` handed a function to `.sort()`.

In [ ]:
# IDEA 1: A FUNCTION IS A VALUE
# ------------------------------------------------------------
# A variable can hold a number, some text, a list... and also a function.
# The function's name WITHOUT brackets is the function itself.
# Adding brackets () is what RUNS it.

def shout(text):
    return text.upper()


print(shout("hello"))    # shout("hello") -> brackets: RUN it and get "HELLO"
# shout          -> no brackets: the function itself, not run
print(shout)

# Because a function is just a value, you can give it a second name.
# yell and shout are now two labels on the SAME function (like b = a on Day 4).
yell = shout
print(yell("hello"))     # runs the same function -> "HELLO"

# And because it's a value, you can hand it to another function as an argument,
# exactly like handing over a number. This is what key=lambda does with .sort().


def run_it(some_function, text):
    # the receiving function decides WHEN to run it
    return some_function(text)


print(run_it(shout, "salaam"))       # pass shout itself - no brackets!

HELLO
<function shout at 0x10a3880f0>
HELLO
SALAAM


### Idea 2: a function can create and return another function

In [ ]:
# IDEA 2: A FUNCTION CAN CREATE AND RETURN ANOTHER FUNCTION
# ------------------------------------------------------------
# make_greeter is a "function factory": give it a greeting,
# and it builds and hands back a brand-new function that uses that greeting.

def make_greeter(greeting):              # step 1: e.g. greeting = "Salaam"

    def greet(name):                     # step 2: CREATE a function inside - it does NOT run yet
        return f"{greeting}, {name}!"    # <-- CLOSURE: greet uses greeting, which belongs to
                                         #     the OUTER function. This line is what makes
                                         #     greet a closure.

    return greet                         # step 3: hand back the closure itself (no brackets!)
                                         #         make_greeter is now finished

say_salaam = make_greeter("Salaam")      # say_salaam holds a greet function that says "Salaam"
say_hello = make_greeter("Hello")        # a SECOND, separate greet function that says "Hello"

print(say_salaam("Ali"))                 # step 4: NOW greet runs, with name = "Ali"
print(say_hello("Fatima"))

# The surprising part: greeting was a LOCAL variable of make_greeter,
# and make_greeter finished long ago. Yesterday you learned that local variables
# disappear when a function ends - so how does greet still know "Salaam"?
# Because greet was created inside make_greeter, it carries the variables it
# needs with it, like a backpack. A function + its backpack = a CLOSURE.

# Proof: every closure really does carry its backpack, and you can peek inside.
# (__closure__ is Python's name for the backpack - you'll rarely need it, it's just to see it.)
print(say_salaam.__closure__[0].cell_contents)    # "Salaam" - still stored inside say_salaam
print(say_hello.__closure__[0].cell_contents)     # "Hello"  - its own, separate backpack

In [15]:
# A business example of Idea 2: a discount "factory".
# Each call to make_discount builds a new function that remembers its own percent.

def make_discount(percent):
    def apply(price):
        return price * (1 - percent / 100)    # <-- CLOSURE: apply uses percent from the
                                              #     outer function, so apply is a closure
    return apply                              # hand back the closure

eid_sale = make_discount(20)       # a function that always takes 20% off
clearance = make_discount(50)      # a function that always takes 50% off

print(eid_sale(1000))              # 800.0
print(clearance(1000))             # 500.0

800.0
500.0


`greet` remembers `greeting` even after `make_greeter` has finished running. A function that remembers values from where it was created is called a **closure** — and it's exactly what makes decorators work.

### Putting it together: a decorator

A decorator is a function that **takes a function, and returns a new version of it** with extra behaviour wrapped around the original.

In [18]:
# PUTTING IT TOGETHER: A DECORATOR = IDEA 1 + IDEA 2
# ------------------------------------------------------------

def announce(func):                  # IDEA 1: announce RECEIVES a function (func) as a value
    def wrapper():                   # IDEA 2: it CREATES a new function inside
        print("About to run...")     #         extra behaviour BEFORE
        func()                       # <-- CLOSURE: wrapper uses func from the outer function,
                                     #     so wrapper is a closure that remembers which
                                     #     function it wrapped
        print("Finished.")           #         extra behaviour AFTER
    return wrapper                   # IDEA 2: it HANDS BACK the closure (no brackets!)

def make_tea():
    print("Making tea")

# announce(make_tea) -> pass make_tea itself in (Idea 1 - no brackets on make_tea)
#                    -> get back wrapper, which remembers make_tea (Idea 2 + closure)
# make_tea = ...     -> the name make_tea now points to the wrapped version
makeing_tea = announce(make_tea)

makeing_tea()     # really runs wrapper: "About to run...", then "Making tea", then "Finished."

About to run...
Making tea
Finished.


`make_tea` itself never changed — `announce` built a new function around it. Writing `make_tea = announce(make_tea)` every time is clumsy, so Python gives you a shortcut: put `@announce` on the line above the `def`.

In [21]:
# The @ line is only a shortcut. Right after the def, Python does:
#     make_coffee = announce(make_coffee)
# The same Idea 1 + Idea 2 as the cell above - just less typing.

@announce
def make_coffee():
    print("Making coffee")

make_coffee()

About to run...
Making coffee
Finished.


### Making a decorator work for any function

`announce` only works on functions with no arguments, because `wrapper()` takes none. To wrap *any* function, the wrapper uses `*args` and `**kwargs` — this is where yesterday's lesson pays off.

In [ ]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished.")
        return result                # don't forget to pass the result back
    return wrapper

@announce
def add(a, b):
    return a + b

@announce
def subtract(a, b):
    return a - b

print(add(3, 4))

Calling add...
add finished.
7


Two new pieces:

- In the `def wrapper(*args, **kwargs)` line, the stars **collect** whatever arguments came in, just like yesterday's `*args` and `**kwargs`.
- In the call `func(*args, **kwargs)`, the stars do the opposite — they **spread** them back out, so the original function receives exactly what the wrapper received.

And `func.__name__` is simply the function's own name, as text.

### Real use case: timing a slow sales report

`import time` loads Python's built-in time tools — imports are covered properly on Day 9. For now, `time.time()` just gives the current time in seconds.

In [25]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.3f} seconds")
        return result
    return wrapper

@timer
def build_sales_report(order_count):
    revenue = 0
    for order_id in range(order_count):
        revenue += 1500              # pretend every order was worth Rs 1,500
    return revenue

print(build_sales_report(1000000))

build_sales_report took 0.026 seconds
1500000000


### Real use case: only managers can issue refunds

In a POS system, a cashier can ring up sales, but refunds need a manager. Instead of pasting the same permission check into every sensitive function — refunds, big discounts, voiding a sale — write it once as a decorator.

In [27]:
current_staff = {"name": "Hassan", "role": "cashier"}

def manager_only(func):
    def wrapper(*args, **kwargs):
        if current_staff["role"] != "manager":
            print(f"Denied: {current_staff['name']} is not a manager.")
            return None
        return func(*args, **kwargs)
    return wrapper

@manager_only
def issue_refund(order_id, amount):
    print(f"Refunded Rs {amount} for order {order_id}")

issue_refund("A-1042", 2500)                 # blocked - Hassan is a cashier
current_staff["role"] = "manager"
issue_refund("A-1042", 2500)                 # allowed

Denied: Hassan is not a manager.
Refunded Rs 2500 for order A-1042


### Real use case: an audit log of every sale

A business must be able to answer "who sold what, and for how much?" A decorator can record every call to a function automatically — so nobody can forget to log a sale.

In [ ]:
audit_log = []

def log_transaction(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        audit_log.append(f"{func.__name__}{args} -> {result}")
        return result
    return wrapper

@log_transaction
def sell(item, price, quantity):
    return price * quantity

sell("Mouse", 1500, 2)
sell("Keyboard", 3500, 1)

for entry in audit_log:
    print(entry)

> **Where you'll meet decorators for real:** web frameworks use them to connect a web address to a function (FastAPI's `@app.post("/sales")`, Flask's `@app.route("/home")`), and testing and AI-tooling libraries use them to register functions. You'll *use* ready-made decorators far more often than you'll write your own — but now you know what the `@` is doing.

### Where you'll see decorators everywhere: FastAPI's `@app.post(...)`

**FastAPI** is one of the most popular Python frameworks for building web APIs — the back end behind an online store, a POS system, or an AI app. A FastAPI program looks like this:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/products")
def list_products():
    return ["Mouse", "Keyboard", "Monitor"]

@app.post("/sales")
def create_sale(item: str, price: int, quantity: int):
    return {"item": item, "total": price * quantity}
```

Those `@app.get(...)` and `@app.post(...)` lines are decorators. Let's take `@app.post("/sales")` apart piece by piece:

- **`app`** — the web app. It keeps a list of every address it knows about, and which function handles each one.
- **`.post`** — the kind of request. `post` means "someone is **sending** new data" (placing an order, recording a sale). `get` means "someone wants to **read** data" (show me the products). There are also `put` (update) and `delete`.
- **`("/sales")`** — the address, the part after the website's name: `myshop.com/sales`.
- **the function below it** — what should run when that request arrives.

So the whole line means: **"when someone sends data to `/sales`, run `create_sale`."**

### Why are there brackets after the decorator?

`@announce` had no brackets, but `@app.post("/sales")` does — because this decorator needs extra information: *which address?* That makes it a decorator **factory** — exactly Idea 2 from earlier:

1. `app.post("/sales")` runs first, and **returns a decorator** that remembers `"/sales"` (a closure).
2. That decorator is then applied to `create_sale`.

In other words, these two are the same:

```python
@app.post("/sales")
def create_sale(...): ...

# is just shorthand for:
create_sale = app.post("/sales")(create_sale)
```

### This decorator doesn't wrap — it *registers*

`@announce` built a wrapper around the function. `@app.post` does something different: it writes the function into the app's list of addresses, then hands the function back **unchanged**. Decorators aren't only for adding behaviour — they're also a neat way to *sign a function up* for something.

You can't run FastAPI in this notebook without installing it, but you don't need to — here's a tiny version of what `@app.post` and `@app.get` do behind the scenes, built from nothing but a dictionary, a closure and `**kwargs`:

In [ ]:
routes = {}                                   # the app's list: (method, address) -> function

def post(path):                               # a decorator FACTORY - takes the address
    def decorator(func):                      # the real decorator - takes the function
        routes[("POST", path)] = func         # register it (the closure remembers path)
        return func                           # hand the function back UNCHANGED
    return decorator

def get(path):
    def decorator(func):
        routes[("GET", path)] = func
        return func
    return decorator

@post("/sales")                               # same as: create_sale = post("/sales")(create_sale)
def create_sale(item, price, quantity):
    return {"item": item, "total": price * quantity}

@get("/products")
def list_products():
    return ["Mouse", "Keyboard", "Monitor"]

for key, func in routes.items():              # what got registered
    print(key, "->", func.__name__)

('POST', '/sales') -> create_sale
('GET', '/products') -> list_products


Nothing has *run* yet — the decorators only filled in the `routes` list. Now pretend requests arrive from customers' browsers. This is the job the web server does for you:

In [ ]:
def handle_request(method, path, **data):
    func = routes.get((method, path))         # look up who handles this address
    if func is None:
        return {"error": "404 Not Found"}     # nobody registered this address
    return func(**data)                       # run the matching function with the request's data

print(handle_request("POST", "/sales", item="Mouse", price=1500, quantity=2))
print(handle_request("GET", "/products"))
print(handle_request("GET", "/refunds"))

print(create_sale("Keyboard", 3500, 1))       # the function itself still works normally

That's the whole trick. Real FastAPI does the same thing with a lot more polish:

- It reads the request's data and matches it to your parameters using the type hints — `item: str`, `price: int` — and rejects a request with a `422` error if something is missing or the wrong type, before your function ever runs.
- It turns whatever your function returns into JSON, the format web browsers and apps understand.
- It answers `404 Not Found` for an address nobody registered, and `405 Method Not Allowed` for a `get` sent to an address that only accepts `post`.
- It builds an interactive page at `/docs` where you can try every address.

The full FastAPI version is saved next to this notebook as `fastapi_routes.py`. To run it on your own computer: `pip install "fastapi[standard]"`, then `fastapi dev fastapi_routes.py`, and open `http://127.0.0.1:8000/docs`.